# OV dual-classifier — split strategies comparison v2

Este notebook sirve para **comparar estrategias de partición** antes de cerrar los resultados del manuscrito.

Compara cuatro escenarios:

1. **`original_separate`**  
   - Victimización: split propio, `stratify=y_victim`, `test_size=0.20`.  
   - Perpetración: split propio, `stratify=y_perp`, `test_size=0.25`.  
   - Overlap: se evalúa en el split de perpetración, aplicando ambos modelos.  
   - Útil para ver si se reproducen resultados cercanos a los notebooks antiguos.  
   - Nota: para overlap puede haber solapamiento entre el test de perpetración y el train de victimización; se marca como escenario exploratorio/histórico.

2. **`common_combined`**  
   - Un único split común, `test_size=0.25`, estratificado por `y_victim + y_perp`.
   - Es el escenario más coherente para evaluar los tres outcomes en los mismos adolescentes.

3. **`common_perp`**  
   - Un único split común, `test_size=0.25`, estratificado solo por perpetración.
   - Compromiso posible si se quiere priorizar estabilidad del outcome más difícil.

4. **`common_victim`**  
   - Un único split común, `test_size=0.25`, estratificado solo por victimización.
   - Sirve como control adicional.

El objetivo no es elegir “el resultado que más gusta”, sino comprobar si las conclusiones son estables o dependen demasiado del split.


In [1]:
# =========================
# 0. Configuración general
# =========================

DATA_DIR = './data'
OUTPUT_DIR = './content/dual_split_comparison_v2'

FEATURES_FILE = f'{DATA_DIR}/lista_global_vars.csv'
TARGET_FILE = f'{DATA_DIR}/target_col.csv'

RANDOM_STATE = 42
PCA_VARIANCE_THRESHOLD = 0.95

# Splits
TEST_SIZE_VICTIM_ORIGINAL = 0.20
TEST_SIZE_PERP_ORIGINAL = 0.25
TEST_SIZE_COMMON = 0.25

# Umbrales iniciales para tabla comparable.
# Después se exploran thresholds con tablas específicas.
VICTIM_THRESHOLD_DEFAULT = 0.25
PERP_THRESHOLD_DEFAULT = 0.50

# Criterios de selección de thresholds exploratorios
MIN_RECALL_VICTIM = 0.90
MIN_RECALL_PERP = 0.90
MIN_RECALL_OVERLAP = 0.90

# Árbol de victimización
VICTIM_CLASS_WEIGHT = None  # Cambiar a 'balanced' solo si se quiere probar explícitamente.

# Red neuronal perpetración
NN_EPOCHS = 200
NN_BATCH_SIZE = 128
NN_PATIENCE = 20
NN_VALIDATION_SPLIT = 0.15

print('Config loaded')


Config loaded


In [2]:
# =========================
# 1. Imports
# =========================

import os
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    classification_report,
    average_precision_score
)

import joblib

os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    from tensorflow.keras.callbacks import EarlyStopping
    TF_AVAILABLE = True
    tf.random.set_seed(RANDOM_STATE)
except Exception as e:
    TF_AVAILABLE = False
    print('TensorFlow no disponible:', e)

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

print('Imports OK')


Imports OK


## 2. Funciones auxiliares


In [3]:
def prepare_dataset(features_file=FEATURES_FILE, target_file=TARGET_FILE):
    """Carga, filtra y prepara el dataset común para ambos modelos."""
    feat_df = pd.read_csv(features_file)
    target_df = pd.read_csv(target_file).fillna(0)

    print('Dim características original:', feat_df.shape)
    print('Dim targets original:', target_df.shape)

    df = feat_df.join(target_df, how='inner')

    # Filtro usado en notebooks previos: elimina categorías muy desbalanceadas.
    filter_mask = ~((df['GENERO_BIN_2'] == 1) | (df['ORIENTSEX.BN_3'] == 1))
    df = df[filter_mask].drop(columns=['GENERO_BIN_2', 'ORIENTSEX.BN_3']).reset_index(drop=True)

    y_victim = df['VÍCTIMA'].astype(int)
    y_perp = df['PERPETRADOR'].astype(int)
    if 'VICTIMA_PERPETRADOR' in df.columns:
        y_overlap = df['VICTIMA_PERPETRADOR'].astype(int)
    else:
        y_overlap = ((y_victim == 1) & (y_perp == 1)).astype(int)

    leakage_cols = [
        'VÍCTIMA', 'PERPETRADOR', 'VICTIMA_PERPETRADOR',
        'POLIVICTIMIZACION', 'POLIPERPETRACION',
        'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'NO.VICT_NO.PERP',
        'V.O', 'P.SUM.TOTAL', 'V.SUM.TOTAL'
    ]
    X = df.drop(columns=[c for c in leakage_cols if c in df.columns]).copy()

    pd.set_option('future.no_silent_downcasting', True)

    # Feature engineering básico de notebooks previos.
    if 'PAÍS' in X.columns:
        X['PAÍS'] = X['PAÍS'].replace({1: True, 2: False})
    if 'ETNIA.BN' in X.columns:
        X['ETNIA.BN'] = X['ETNIA.BN'].replace({0.0: False, 1.0: True})
    if 'FUGAS.BN' in X.columns:
        X['FUGAS.BN'] = X['FUGAS.BN'].replace({0.0: False, 1.0: True})

    if {'GENERO_BIN_0', 'GENERO_BIN_1'}.issubset(X.columns):
        X['GENERO.BN0'] = X['GENERO_BIN_0'].replace({0.0: False, 1.0: True})
        X['GENERO.BN1'] = X['GENERO_BIN_1'].replace({0.0: False, 1.0: True})
        X = X.drop(columns=['GENERO_BIN_0', 'GENERO_BIN_1'])

    if {'ORIENTSEX.BN_1', 'ORIENTSEX.BN_2'}.issubset(X.columns):
        X['ORIENTSEX.BN0'] = X['ORIENTSEX.BN_1'].replace({0.0: False, 1.0: True})
        X['ORIENTSEX.BN1'] = X['ORIENTSEX.BN_2'].replace({0.0: False, 1.0: True})
        X = X.drop(columns=['ORIENTSEX.BN_1', 'ORIENTSEX.BN_2'])

    if 'CONVIVEN.5' in X.columns:
        X = X.rename(columns={'CONVIVEN.5': 'CONVIVEN_H'})
        X['CONVIVEN_H'] = X['CONVIVEN_H'].replace({0.0: False, 1.0: True})
    if 'CONVIVEN.6' in X.columns:
        X = X.rename(columns={'CONVIVEN.6': 'CONVIVEN_0'})
        X['CONVIVEN_0'] = X['CONVIVEN_0'].replace({0.0: False, 1.0: True})

    bool_cols = X.select_dtypes(include=['bool']).columns
    X[bool_cols] = X[bool_cols].astype(int)
    X = X.apply(pd.to_numeric, errors='raise')

    print('Dataset final:', X.shape)
    print('Targets:', len(y_victim), len(y_perp), len(y_overlap))
    print('Missings X:', int(X.isna().sum().sum()))

    return X, y_victim, y_perp, y_overlap, df


def print_distribution(name, y):
    vc = y.value_counts().sort_index()
    pct = y.value_counts(normalize=True).sort_index() * 100
    out = pd.DataFrame({'n': vc, '%': pct.round(2)})
    print('' + name)
    display(out)
    return out


def describe_indices(name, idx, y_victim, y_perp, y_overlap):
    print('' + '=' * 100)
    print(name)
    print('N:', len(idx))
    print_distribution('Victimization', y_victim.loc[idx])
    print_distribution('Perpetration', y_perp.loc[idx])
    print_distribution('Overlap', y_overlap.loc[idx])
    print('Victim x Perp')
    display(pd.crosstab(
        y_victim.loc[idx],
        y_perp.loc[idx],
        rownames=['Victim'],
        colnames=['Perpetrator'],
        margins=True
    ))


def select_n_components(pca, threshold=PCA_VARIANCE_THRESHOLD):
    cum = np.cumsum(pca.explained_variance_ratio_)
    return int(np.searchsorted(cum, threshold) + 1)


def fit_pca_for_indices(X, train_idx, eval_idx, prefix=''):
    """Ajusta scaler + PCA solo en train y transforma train/eval."""
    X_train_raw = X.loc[train_idx]
    X_eval_raw = X.loc[eval_idx]

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_eval_scaled = scaler.transform(X_eval_raw)

    train_means = X_train_scaled.mean(axis=0)
    X_train_centered = X_train_scaled - train_means
    X_eval_centered = X_eval_scaled - train_means

    pca_full = PCA(n_components=X_train_raw.shape[1])
    X_train_pca_full = pca_full.fit_transform(X_train_centered)
    X_eval_pca_full = pca_full.transform(X_eval_centered)

    n_components = select_n_components(pca_full, PCA_VARIANCE_THRESHOLD)
    pc_cols = [f'PC{i+1}' for i in range(n_components)]

    X_train_pca = pd.DataFrame(X_train_pca_full[:, :n_components], index=train_idx, columns=pc_cols)
    X_eval_pca = pd.DataFrame(X_eval_pca_full[:, :n_components], index=eval_idx, columns=pc_cols)

    meta = {
        'scaler': scaler,
        'train_means': train_means,
        'pca': pca_full,
        'n_components': n_components,
        'explained_variance_cum': float(np.cumsum(pca_full.explained_variance_ratio_)[n_components - 1]),
        'feature_names': list(X.columns),
        'pc_cols': pc_cols,
        'prefix': prefix
    }
    return X_train_pca, X_eval_pca, meta


def transform_with_pca(X, eval_idx, meta):
    """Transforma nuevos índices usando scaler + PCA ya ajustados."""
    X_eval_raw = X.loc[eval_idx]
    X_eval_scaled = meta['scaler'].transform(X_eval_raw)
    X_eval_centered = X_eval_scaled - meta['train_means']
    X_eval_pca_full = meta['pca'].transform(X_eval_centered)
    n_components = meta['n_components']
    return pd.DataFrame(X_eval_pca_full[:, :n_components], index=eval_idx, columns=meta['pc_cols'])


def evaluate_binary(y_true, y_pred, y_prob=None, label='model'):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) else np.nan
    npv = tn / (tn + fn) if (tn + fn) else np.nan
    row = {
        'Outcome': label,
        'N_test': len(y_true),
        'Accuracy': accuracy_score(y_true, y_pred),
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred),
        'Recall (Class 1)': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': specificity,
        'Precision (PPV)': precision_score(y_true, y_pred, zero_division=0),
        'NPV': npv,
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'TP': int(tp),
        'FP': int(fp),
        'TN': int(tn),
        'FN': int(fn),
    }
    if y_prob is not None:
        row['AUC-PR'] = average_precision_score(y_true, y_prob)
    return row


def pct_table(df):
    cols_pct = ['Accuracy', 'Balanced Accuracy', 'Recall (Class 1)', 'Specificity', 'Precision (PPV)', 'NPV', 'F1']
    out = df.copy()
    for c in cols_pct:
        out[c] = (out[c] * 100).round(1).astype(str) + '%'
    if 'AUC-PR' in out.columns:
        out['AUC-PR'] = out['AUC-PR'].round(3)
    return out


def evaluate_thresholds_from_prob(y_true, y_prob, thresholds):
    rows = []
    for threshold in thresholds:
        y_pred = (y_prob >= threshold).astype(int)
        rows.append({'threshold': float(threshold), **evaluate_binary(y_true, y_pred, y_prob=None)})
    df_raw = pd.DataFrame(rows).drop(columns=['Outcome'])
    df_display = df_raw.copy()
    for c in ['Accuracy', 'Balanced Accuracy', 'Recall (Class 1)', 'Specificity', 'Precision (PPV)', 'NPV', 'F1']:
        df_display[c] = (df_display[c] * 100).round(1)
    df_display['threshold'] = df_display['threshold'].round(3)
    return df_raw, df_display


## 3. Carga y validación descriptiva del dataset


In [7]:
X, y_victim, y_perp, y_overlap, df_full = prepare_dataset()

print_distribution('Victimization total', y_victim)
print_distribution('Perpetration total', y_perp)
print_distribution('Overlap total', y_overlap)

print('Tabla 4 grupos total')
display(pd.crosstab(y_victim, y_perp, rownames=['Victim'], colnames=['Perpetrator'], margins=True))

X.to_csv(os.path.join(OUTPUT_DIR, 'X_features_clean.csv'), index=True)
pd.DataFrame({
    'y_victim': y_victim,
    'y_perpetrator': y_perp,
    'y_overlap': y_overlap
}).to_csv(os.path.join(OUTPUT_DIR, 'targets.csv'), index=True)


Dim características original: (4024, 29)
Dim targets original: (4024, 11)
Dataset final: (3767, 27)
Targets: 3767 3767 3767
Missings X: 0
Victimization total


,n,%
VÍCTIMA,,
0,1906,50.6
1,1861,49.4


Perpetration total


,n,%
PERPETRADOR,,
0,2882,76.51
1,885,23.49


Overlap total


,n,%
VICTIMA_PERPETRADOR,,
0,3048,80.91
1,719,19.09


Tabla 4 grupos total


Perpetrator,0,1,All
Victim,,,
0,1734,172,1906
1,1148,713,1861
All,2882,885,3767


## 4. Crear y describir los splits a comparar


In [8]:
def make_split_strategies(X, y_victim, y_perp, random_state=RANDOM_STATE):
    idx_all = X.index
    strategies = {}

    # 1. Original separado: victimización 20%, perpetración 25%.
    v_train, v_test = train_test_split(
        idx_all,
        test_size=TEST_SIZE_VICTIM_ORIGINAL,
        random_state=random_state,
        stratify=y_victim
    )
    p_train, p_test = train_test_split(
        idx_all,
        test_size=TEST_SIZE_PERP_ORIGINAL,
        random_state=random_state,
        stratify=y_perp
    )
    strategies['original_separate'] = {
        'description': 'Victim split: stratify=y_victim, test=0.20; Perp split: stratify=y_perp, test=0.25; Overlap evaluated on perp test.',
        'victim_train': v_train,
        'victim_test': v_test,
        'perp_train': p_train,
        'perp_test': p_test,
        'overlap_test': p_test,
        'exploratory_warning': True
    }

    # 2. Común combinado.
    stratify_combined = y_victim.astype(str) + '_' + y_perp.astype(str)
    c_train, c_test = train_test_split(
        idx_all,
        test_size=TEST_SIZE_COMMON,
        random_state=random_state,
        stratify=stratify_combined
    )
    strategies['common_combined'] = {
        'description': 'Common split: stratify=y_victim_y_perp, test=0.25.',
        'victim_train': c_train,
        'victim_test': c_test,
        'perp_train': c_train,
        'perp_test': c_test,
        'overlap_test': c_test,
        'exploratory_warning': False
    }

    # 3. Común estratificado por perpetración.
    cp_train, cp_test = train_test_split(
        idx_all,
        test_size=TEST_SIZE_COMMON,
        random_state=random_state,
        stratify=y_perp
    )
    strategies['common_perp'] = {
        'description': 'Common split: stratify=y_perp, test=0.25.',
        'victim_train': cp_train,
        'victim_test': cp_test,
        'perp_train': cp_train,
        'perp_test': cp_test,
        'overlap_test': cp_test,
        'exploratory_warning': False
    }

    # 4. Común estratificado por victimización.
    cv_train, cv_test = train_test_split(
        idx_all,
        test_size=TEST_SIZE_COMMON,
        random_state=random_state,
        stratify=y_victim
    )
    strategies['common_victim'] = {
        'description': 'Common split: stratify=y_victim, test=0.25.',
        'victim_train': cv_train,
        'victim_test': cv_test,
        'perp_train': cv_train,
        'perp_test': cv_test,
        'overlap_test': cv_test,
        'exploratory_warning': False
    }

    return strategies

splits = make_split_strategies(X, y_victim, y_perp)

# Guardar índices
serializable = {}
for name, s in splits.items():
    serializable[name] = {
        'description': s['description'],
        'victim_train': list(map(int, s['victim_train'])),
        'victim_test': list(map(int, s['victim_test'])),
        'perp_train': list(map(int, s['perp_train'])),
        'perp_test': list(map(int, s['perp_test'])),
        'overlap_test': list(map(int, s['overlap_test'])),
        'exploratory_warning': s['exploratory_warning']
    }

with open(os.path.join(OUTPUT_DIR, 'split_strategies_indices.json'), 'w', encoding='utf-8') as f:
    json.dump(serializable, f, ensure_ascii=False, indent=2)

for name, s in splits.items():
    print('' + '#' * 120)
    print(name)
    print(s['description'])
    if s['exploratory_warning']:
        print('WARNING: escenario exploratorio/histórico; revisar posible leakage en overlap si se aplica victim model sobre perp test.')
    describe_indices('Victim test', s['victim_test'], y_victim, y_perp, y_overlap)
    if not np.array_equal(np.sort(s['victim_test']), np.sort(s['perp_test'])):
        describe_indices('Perp / overlap test', s['perp_test'], y_victim, y_perp, y_overlap)


########################################################################################################################
original_separate
Victim split: stratify=y_victim, test=0.20; Perp split: stratify=y_perp, test=0.25; Overlap evaluated on perp test.
Victim test
N: 754
Victimization


,n,%
VÍCTIMA,,
0,382,50.66
1,372,49.34


Perpetration


,n,%
PERPETRADOR,,
0,570,75.6
1,184,24.4


Overlap


...

,n,%
PERPETRADOR,,
0,721,76.54
1,221,23.46


Overlap


,n,%
VICTIMA_PERPETRADOR,,
0,768,81.53
1,174,18.47


Victim x Perp


Perpetrator,0,1,All
Victim,,,
0,430,47,477
1,291,174,465
All,721,221,942


## 5. Entrenamiento y evaluación por escenario

La función siguiente entrena:

- árbol de victimización con cost-complexity pruning;
- red neuronal de perpetración;
- overlap como `Smart AND`.

Además guarda:

- métricas con thresholds por defecto;
- tabla de thresholds de victimización;
- tabla de thresholds de perpetración;
- grid de thresholds para overlap.


In [11]:
def build_perp_model(input_dim, random_state=RANDOM_STATE):
    if not TF_AVAILABLE:
        raise ImportError('TensorFlow no está disponible. Instala tensorflow para entrenar la red neuronal.')
    tf.keras.backend.clear_session()
    tf.random.set_seed(random_state)
    np.random.seed(random_state)
    random.seed(random_state)

    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.20),
        layers.Dense(16, activation='relu'),
        layers.Dropout(0.20),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=[
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.BinaryAccuracy(name='accuracy')
        ]
    )
    return model


def train_victim_tree_for_split(X_train_pca, X_test_pca, y_train, y_test, threshold=VICTIM_THRESHOLD_DEFAULT, scenario_dir=None):
    full_tree = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight=VICTIM_CLASS_WEIGHT)
    full_tree.fit(X_train_pca, y_train)
    path = full_tree.cost_complexity_pruning_path(X_train_pca, y_train)
    ccp_alphas = np.unique(path.ccp_alphas)

    rows = []
    for alpha in ccp_alphas:
        clf = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=alpha, class_weight=VICTIM_CLASS_WEIGHT)
        clf.fit(X_train_pca, y_train)
        prob = clf.predict_proba(X_test_pca)[:, 1]
        pred = (prob >= threshold).astype(int)
        rows.append({
            'ccp_alpha': alpha,
            'recall': recall_score(y_test, pred, zero_division=0),
            'specificity': recall_score(1 - y_test, 1 - pred, zero_division=0),
            'precision': precision_score(y_test, pred, zero_division=0),
            'f1': f1_score(y_test, pred, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_test, pred),
            'depth': clf.get_depth(),
            'n_leaves': clf.get_n_leaves()
        })

    alpha_results = pd.DataFrame(rows)
    candidates = alpha_results[alpha_results['recall'] >= MIN_RECALL_VICTIM].copy()
    if len(candidates) == 0:
        best_row = alpha_results.sort_values(['recall', 'balanced_accuracy', 'specificity'], ascending=False).iloc[0]
    else:
        best_row = candidates.sort_values(['balanced_accuracy', 'specificity', 'precision'], ascending=False).iloc[0]

    best_alpha = float(best_row['ccp_alpha'])
    victim_model = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=best_alpha, class_weight=VICTIM_CLASS_WEIGHT)
    victim_model.fit(X_train_pca, y_train)

    prob = victim_model.predict_proba(X_test_pca)[:, 1]
    pred = (prob >= threshold).astype(int)

    if scenario_dir:
        alpha_results.to_csv(os.path.join(scenario_dir, 'victim_tree_alpha_results.csv'), index=False)
        with open(os.path.join(scenario_dir, 'victim_tree_rules.txt'), 'w', encoding='utf-8') as f:
            f.write(export_text(victim_model, feature_names=list(X_train_pca.columns)))
        joblib.dump(victim_model, os.path.join(scenario_dir, 'victim_decision_tree.joblib'))

    return victim_model, prob, pred, alpha_results, best_row


def train_perp_nn_for_split(X_train_pca, X_test_pca, y_train, y_test, threshold=PERP_THRESHOLD_DEFAULT, scenario_dir=None, verbose=0):
    neg, pos = np.bincount(y_train.astype(int))
    total = neg + pos
    class_weight = {0: total / (2 * neg), 1: total / (2 * pos)}

    model = build_perp_model(X_train_pca.shape[1])
    history = model.fit(
        X_train_pca.values,
        y_train.values,
        validation_split=NN_VALIDATION_SPLIT,
        epochs=NN_EPOCHS,
        batch_size=NN_BATCH_SIZE,
        class_weight=class_weight,
        callbacks=[EarlyStopping(monitor='val_recall', mode='max', patience=NN_PATIENCE, restore_best_weights=True)],
        verbose=verbose
    )

    prob = model.predict(X_test_pca.values, verbose=0).ravel()
    pred = (prob >= threshold).astype(int)

    if scenario_dir:
        model.save(os.path.join(scenario_dir, 'perpetration_nn.keras'))
        hist_df = pd.DataFrame(history.history)
        hist_df.to_csv(os.path.join(scenario_dir, 'perpetration_nn_history.csv'), index=False)
        plt.figure(figsize=(8, 5))
        plt.plot(history.history.get('recall', []), label='train recall')
        plt.plot(history.history.get('val_recall', []), label='val recall')
        plt.xlabel('Epoch')
        plt.ylabel('Recall')
        plt.title('Perpetration NN recall')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(scenario_dir, 'perpetration_nn_recall_history.png'), dpi=300)
        plt.close()

    return model, prob, pred, history, class_weight


def run_overlap_grid(y_overlap_true, victim_prob_for_overlap, perp_prob_for_overlap,
                     victim_thresholds=None, perp_thresholds=None):
    if victim_thresholds is None:
        victim_thresholds = [0.25, 0.26, 0.27, 0.30, 0.35, 0.40, 0.50]
    if perp_thresholds is None:
        perp_thresholds = np.arange(0.10, 0.91, 0.05)
    rows = []
    for vt in victim_thresholds:
        for pt in perp_thresholds:
            vp = (victim_prob_for_overlap >= vt).astype(int)
            pp = (perp_prob_for_overlap >= pt).astype(int)
            op = ((vp == 1) & (pp == 1)).astype(int)
            rows.append({
                'victim_threshold': float(vt),
                'perp_threshold': float(pt),
                **evaluate_binary(y_overlap_true, op, None, label='Overlap')
            })
    raw = pd.DataFrame(rows)
    display_df = raw.copy()
    for c in ['Accuracy', 'Balanced Accuracy', 'Recall (Class 1)', 'Specificity', 'Precision (PPV)', 'NPV', 'F1']:
        display_df[c] = (display_df[c] * 100).round(1)
    display_df['victim_threshold'] = display_df['victim_threshold'].round(3)
    display_df['perp_threshold'] = display_df['perp_threshold'].round(3)
    return raw, display_df


def run_scenario(name, split, verbose_nn=0):
    scenario_dir = os.path.join(OUTPUT_DIR, name)
    os.makedirs(scenario_dir, exist_ok=True)
    print('' + '=' * 120)
    print('RUNNING SCENARIO:', name)
    print(split['description'])
    if split.get('exploratory_warning'):
        print('WARNING: escenario exploratorio/histórico. Revisar posible leakage en overlap.')

    # -------------------------
    # Victim model
    # -------------------------
    Xv_train_pca, Xv_test_pca, victim_pca_meta = fit_pca_for_indices(X, split['victim_train'], split['victim_test'], prefix=f'{name}_victim')
    yv_train = y_victim.loc[split['victim_train']]
    yv_test = y_victim.loc[split['victim_test']]

    victim_model, victim_prob, victim_pred, victim_alpha_results, victim_best_row = train_victim_tree_for_split(
        Xv_train_pca, Xv_test_pca, yv_train, yv_test,
        threshold=VICTIM_THRESHOLD_DEFAULT,
        scenario_dir=scenario_dir
    )

    print('Victimization — default threshold')
    print('Best alpha:', float(victim_best_row['ccp_alpha']))
    print('Depth:', victim_model.get_depth(), 'Leaves:', victim_model.get_n_leaves())
    print(classification_report(yv_test, victim_pred, digits=3))

    # Threshold table victim
    victim_thresholds_raw, victim_thresholds = evaluate_thresholds_from_prob(
        yv_test, victim_prob, thresholds=np.arange(0.10, 0.91, 0.05)
    )
    victim_thresholds_raw.to_csv(os.path.join(scenario_dir, 'victim_thresholds_raw.csv'), index=False)
    victim_thresholds.to_csv(os.path.join(scenario_dir, 'victim_thresholds_percent.csv'), index=False)

    # -------------------------
    # Perp model
    # -------------------------
    Xp_train_pca, Xp_test_pca, perp_pca_meta = fit_pca_for_indices(X, split['perp_train'], split['perp_test'], prefix=f'{name}_perp')
    yp_train = y_perp.loc[split['perp_train']]
    yp_test = y_perp.loc[split['perp_test']]

    perp_model, perp_prob, perp_pred, history, class_weight = train_perp_nn_for_split(
        Xp_train_pca, Xp_test_pca, yp_train, yp_test,
        threshold=PERP_THRESHOLD_DEFAULT,
        scenario_dir=scenario_dir,
        verbose=verbose_nn
    )

    print('Perpetration — default threshold')
    print('Class weight:', class_weight)
    print(classification_report(yp_test, perp_pred, digits=3))

    # Threshold table perpetration
    perp_thresholds_raw, perp_thresholds = evaluate_thresholds_from_prob(
        yp_test, perp_prob, thresholds=np.arange(0.10, 0.91, 0.05)
    )
    perp_thresholds_raw.to_csv(os.path.join(scenario_dir, 'perp_thresholds_raw.csv'), index=False)
    perp_thresholds.to_csv(os.path.join(scenario_dir, 'perp_thresholds_percent.csv'), index=False)

    # -------------------------
    # Overlap on split['overlap_test']
    # -------------------------
    overlap_idx = split['overlap_test']
    yo_test = y_overlap.loc[overlap_idx]

    Xv_overlap_pca = transform_with_pca(X, overlap_idx, victim_pca_meta)
    Xp_overlap_pca = transform_with_pca(X, overlap_idx, perp_pca_meta)

    victim_prob_overlap = victim_model.predict_proba(Xv_overlap_pca)[:, 1]
    perp_prob_overlap = perp_model.predict(Xp_overlap_pca.values, verbose=0).ravel()

    victim_pred_overlap_base = (victim_prob_overlap >= VICTIM_THRESHOLD_DEFAULT).astype(int)
    perp_pred_overlap_base = (perp_prob_overlap >= PERP_THRESHOLD_DEFAULT).astype(int)
    overlap_pred = ((victim_pred_overlap_base == 1) & (perp_pred_overlap_base == 1)).astype(int)
    overlap_prob = np.minimum(victim_prob_overlap, perp_prob_overlap)

    print('Overlap — default thresholds')
    print(classification_report(yo_test, overlap_pred, digits=3))

    overlap_grid_raw, overlap_grid = run_overlap_grid(yo_test, victim_prob_overlap, perp_prob_overlap)
    overlap_grid_raw.to_csv(os.path.join(scenario_dir, 'overlap_threshold_grid_raw.csv'), index=False)
    overlap_grid.to_csv(os.path.join(scenario_dir, 'overlap_threshold_grid_percent.csv'), index=False)

    # -------------------------
    # Final default metrics table
    # -------------------------
    metrics_rows = [
        evaluate_binary(yv_test, victim_pred, victim_prob, label='Victimization'),
        evaluate_binary(yp_test, perp_pred, perp_prob, label='Perpetration'),
        evaluate_binary(yo_test, overlap_pred, overlap_prob, label='Overlap')
    ]
    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.insert(1, 'Model', ['Decision Tree', 'Neural Network (DNN)', 'Smart AND'])
    metrics_df.insert(0, 'Scenario', name)

    metrics_df.to_csv(os.path.join(scenario_dir, 'metrics_default_raw.csv'), index=False)
    metrics_pct = pct_table(metrics_df)
    metrics_pct.to_csv(os.path.join(scenario_dir, 'metrics_default_percent.csv'), index=False)

    # Predictions
    pd.DataFrame({
        'ori_idx': split['victim_test'],
        'y_true_victim': yv_test.values,
        'y_pred_victim': victim_pred,
        'prob_victim': victim_prob
    }).set_index('ori_idx').to_csv(os.path.join(scenario_dir, 'victim_predictions.csv'))

    pd.DataFrame({
        'ori_idx': split['perp_test'],
        'y_true_perpetrator': yp_test.values,
        'y_pred_perpetrator': perp_pred,
        'prob_perpetrator': perp_prob
    }).set_index('ori_idx').to_csv(os.path.join(scenario_dir, 'perp_predictions.csv'))

    pd.DataFrame({
        'ori_idx': overlap_idx,
        'y_true_overlap': yo_test.values,
        'y_pred_overlap': overlap_pred,
        'prob_overlap_min': overlap_prob,
        'y_pred_victim': victim_pred_overlap_base,
        'prob_victim': victim_prob_overlap,
        'y_pred_perpetrator': perp_pred_overlap_base,
        'prob_perpetrator': perp_prob_overlap
    }).set_index('ori_idx').to_csv(os.path.join(scenario_dir, 'overlap_predictions.csv'))

    result = {
        'scenario': name,
        'metrics_raw': metrics_df,
        'metrics_percent': metrics_pct,
        'victim_thresholds_raw': victim_thresholds_raw,
        'victim_thresholds_percent': victim_thresholds,
        'perp_thresholds_raw': perp_thresholds_raw,
        'perp_thresholds_percent': perp_thresholds,
        'overlap_grid_raw': overlap_grid_raw,
        'overlap_grid_percent': overlap_grid,
        'victim_model': victim_model,
        'perp_model': perp_model,
        'victim_pca_meta': victim_pca_meta,
        'perp_pca_meta': perp_pca_meta,
        'victim_best_alpha_row': victim_best_row,
        'scenario_dir': scenario_dir
    }
    return result


## 6. Ejecutar escenarios

Puedes ejecutar todos, o solo uno cambiando la lista `SCENARIOS_TO_RUN`.

Para ir rápido al principio, puedes empezar con:

```python
SCENARIOS_TO_RUN = ['original_separate', 'common_combined']
```

Cuando tengas tiempo, ejecuta los cuatro.


In [12]:
SCENARIOS_TO_RUN = [
    'original_separate',
    'common_combined',
    'common_perp',
    'common_victim'
]

results = {}
for scenario_name in SCENARIOS_TO_RUN:
    results[scenario_name] = run_scenario(scenario_name, splits[scenario_name], verbose_nn=0)

print('Scenarios completed:', list(results.keys()))


RUNNING SCENARIO: original_separate
Victim split: stratify=y_victim, test=0.20; Perp split: stratify=y_perp, test=0.25; Overlap evaluated on perp test.
Victimization — default threshold
Best alpha: 0.0017816596233200116
Depth: 7 Leaves: 15
              precision    recall  f1-score   support

           0      0.744     0.251     0.376       382
           1      0.542     0.911     0.680       372

    accuracy                          0.577       754
   macro avg      0.643     0.581     0.528       754
weighted avg      0.645     0.577     0.526       754

Perpetration — default threshold
Class weight: {0: np.float64(0.6536325775104118), 1: np.float64(2.1272590361445785)}
              precision    recall  f1-score   support

           0      0.853     0.675     0.754       721
           1      0.369     0.620     0.463       221

    accuracy                          0.662       942
   macro avg      0.611     0.648     0.608       942
weighted avg      0.739     0.662     0.686

## 7. Tabla comparativa global con thresholds por defecto

Esta tabla compara los escenarios usando:

- victim threshold = `VICTIM_THRESHOLD_DEFAULT`
- perpetration threshold = `PERP_THRESHOLD_DEFAULT`
- overlap = Smart AND con esos dos thresholds


In [13]:
comparison_raw = pd.concat([results[name]['metrics_raw'] for name in results], ignore_index=True)
comparison_pct = pct_table(comparison_raw)

comparison_raw.to_csv(os.path.join(OUTPUT_DIR, 'comparison_all_scenarios_default_raw.csv'), index=False)
comparison_pct.to_csv(os.path.join(OUTPUT_DIR, 'comparison_all_scenarios_default_percent.csv'), index=False)

display(comparison_pct)


,Scenario,Outcome,Model,N_test,Accuracy,Balanced Accuracy,Recall (Class 1),Specificity,Precision (PPV),NPV,F1,TP,FP,TN,FN,AUC-PR
0,original_separate,Victimization,Decision Tree,754,57.7%,58.1%,91.1%,25.1%,54.2%,74.4%,68.0%,339,286,96,33,0.659
1,original_separate,Perpetration,Neural Network (DNN),942,66.2%,64.8%,62.0%,67.5%,36.9%,85.3%,46.3%,137,234,487,84,0.430
2,original_separate,Overlap,Smart AND,942,66.1%,65.2%,63.7%,66.7%,31.0%,88.7%,41.7%,114,254,509,65,0.374
3,common_combined,Victimization,Decision Tree,942,55.8%,56.3%,94.0%,18.7%,53.0%,76.1%,67.8%,437,388,89,28,0.672
4,common_combined,Perpetration,Neural Network (DNN),942,64.6%,64.2%,63.3%,65.0%,35.7%,85.3%,45.7%,140,252,469,81,0.408
5,common_combined,Overlap,Smart AND,942,65.4%,65.9%,66.7%,65.1%,31.1%,89.2%,42.4%,120,266,496,60,0.432
6,common_perp,Victimization,Decision Tree,942,55.5%,56.1%,91.6%,20.7%,52.7%,71.7%,66.9%,424,380,99,39,0.657
7,common_perp,Perpetration,Neural Network (DNN),942,66.2%,64.8%,62.0%,67.5%,36.9%,85.3%,46.3%,137,234,487,84,0.430
8,common_perp,Overlap,Smart AND,942,66.1%,65.2%,63.7%,66.7%,31.0%,88.7%,41.7%,114,254,509,65,0.375
9,common_victim,Victimization,Decision Tree,942,56.3%,56.7%,93.1%,20.3%,53.3%,75.2%,67.8%,433,380,97,32,0.661


## 8. Mejores thresholds por escenario

Estas tablas sirven para ver si el mal rendimiento se debe al split o simplemente a un threshold poco adecuado.


In [ ]:
def best_threshold_table(thresholds_raw, min_recall, sort_cols=None):
    if sort_cols is None:
        sort_cols = ['Balanced Accuracy', 'Specificity', 'Precision (PPV)']
    candidates = thresholds_raw[thresholds_raw['Recall (Class 1)'] >= min_recall].copy()
    if len(candidates) == 0:
        candidates = thresholds_raw.copy()
        candidates['criterion_note'] = f'No threshold reached recall >= {min_recall:.2f}; sorted by recall first.'
        candidates = candidates.sort_values(['Recall (Class 1)', 'Balanced Accuracy', 'Specificity'], ascending=False)
    else:
        candidates['criterion_note'] = f'Recall >= {min_recall:.2f}'
        candidates = candidates.sort_values(sort_cols, ascending=False)
    return candidates

best_rows = []
for name, res in results.items():
    # Victim best
    vbest = best_threshold_table(res['victim_thresholds_raw'], MIN_RECALL_VICTIM).iloc[0].copy()
    vbest['Scenario'] = name
    vbest['Outcome'] = 'Victimization'
    best_rows.append(vbest)

    # Perp best
    pbest = best_threshold_table(res['perp_thresholds_raw'], MIN_RECALL_PERP).iloc[0].copy()
    pbest['Scenario'] = name
    pbest['Outcome'] = 'Perpetration'
    best_rows.append(pbest)

best_thresholds_summary = pd.DataFrame(best_rows)
cols_first = ['Scenario', 'Outcome', 'threshold', 'criterion_note']
best_thresholds_summary = best_thresholds_summary[cols_first + [c for c in best_thresholds_summary.columns if c not in cols_first]]

best_thresholds_summary.to_csv(os.path.join(OUTPUT_DIR, 'best_thresholds_victim_perp_raw.csv'), index=False)

best_thresholds_summary_pct = best_thresholds_summary.copy()
for c in ['Accuracy', 'Balanced Accuracy', 'Recall (Class 1)', 'Specificity', 'Precision (PPV)', 'NPV', 'F1']:
    best_thresholds_summary_pct[c] = (best_thresholds_summary_pct[c] * 100).round(1)

best_thresholds_summary_pct.to_csv(os.path.join(OUTPUT_DIR, 'best_thresholds_victim_perp_percent.csv'), index=False)
display(best_thresholds_summary_pct)


## 9. Mejores combinaciones de thresholds para overlap

El overlap depende de dos thresholds:

```python
victim_pred = prob_victim >= vt
perp_pred = prob_perp >= pt
overlap_pred = victim_pred AND perp_pred
```

Por eso se evalúa un grid de combinaciones.


In [ ]:
overlap_best_rows = []
for name, res in results.items():
    grid = res['overlap_grid_raw'].copy()
    candidates = grid[grid['Recall (Class 1)'] >= MIN_RECALL_OVERLAP].copy()
    if len(candidates) == 0:
        candidates = grid.copy()
        candidates['criterion_note'] = f'No overlap threshold pair reached recall >= {MIN_RECALL_OVERLAP:.2f}; sorted by recall first.'
        candidates = candidates.sort_values(['Recall (Class 1)', 'Balanced Accuracy', 'Specificity'], ascending=False)
    else:
        candidates['criterion_note'] = f'Overlap recall >= {MIN_RECALL_OVERLAP:.2f}'
        candidates = candidates.sort_values(['Balanced Accuracy', 'Specificity', 'Precision (PPV)'], ascending=False)
    best = candidates.iloc[0].copy()
    best['Scenario'] = name
    overlap_best_rows.append(best)

overlap_best_summary = pd.DataFrame(overlap_best_rows)
cols_first = ['Scenario', 'victim_threshold', 'perp_threshold', 'criterion_note']
overlap_best_summary = overlap_best_summary[cols_first + [c for c in overlap_best_summary.columns if c not in cols_first]]
overlap_best_summary.to_csv(os.path.join(OUTPUT_DIR, 'best_overlap_threshold_pairs_raw.csv'), index=False)

overlap_best_summary_pct = overlap_best_summary.copy()
for c in ['Accuracy', 'Balanced Accuracy', 'Recall (Class 1)', 'Specificity', 'Precision (PPV)', 'NPV', 'F1']:
    overlap_best_summary_pct[c] = (overlap_best_summary_pct[c] * 100).round(1)

overlap_best_summary_pct.to_csv(os.path.join(OUTPUT_DIR, 'best_overlap_threshold_pairs_percent.csv'), index=False)
display(overlap_best_summary_pct)


## 10. Ver tablas completas de un escenario concreto

Cambia `SCENARIO_TO_INSPECT` para revisar los thresholds de cada modelo.


In [ ]:
SCENARIO_TO_INSPECT = 'common_combined'

print('Scenario:', SCENARIO_TO_INSPECT)
print('Default metrics')
display(results[SCENARIO_TO_INSPECT]['metrics_percent'])

print('Victim thresholds')
display(results[SCENARIO_TO_INSPECT]['victim_thresholds_percent'])

print('Perpetration thresholds')
display(results[SCENARIO_TO_INSPECT]['perp_thresholds_percent'])

print('Top overlap threshold combinations by recall / balanced accuracy')
display(
    results[SCENARIO_TO_INSPECT]['overlap_grid_percent']
    .sort_values(['Recall (Class 1)', 'Balanced Accuracy', 'Specificity'], ascending=False)
    .head(25)
)


## 11. Interpretación metodológica provisional

Usa esta guía al revisar las salidas:

- Si `original_separate` reproduce los resultados antiguos, pero los splits comunes no, el modelo es sensible a la partición.
- Si `common_combined` cae mucho en perpetración, revisar primero thresholds de perpetración.
- Si ningún threshold logra recall alto en perpetración con splits comunes, el claim de “high-sensitivity screening” debe matizarse o habrá que comparar otros modelos.
- Si `common_perp` mejora perpetración/overlap manteniendo un único test, puede ser un compromiso metodológico defendible.
- Si los resultados varían mucho entre escenarios, conviene plantearse repeated splits o cross-validation para reportar media ± desviación típica.
